# Generating type 4 clones with LLMs

### Configuration

In [3]:
import src.clone_gen as cg
LLama3 = "llama3.1:latest" # used for generation of additional fields 
# these are used for the experiment
DeepSeek = "deepseek-r1:14b" 
Gemma3 = "gemma3:latest" 
Gpt20b= "gpt-oss:20b"


DATASET_PATH = "../dataset/dataset.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL =  DeepSeek
NL_MODEL = LLama3    
CODE_MODEL = LLama3    

# Generation settings
LLM_OPTS = {
    "temperature": 0.5,      # lower = more deterministic, higher = more creative
    "top_p": 0.95,            # nucleus sampling: consider only tokens with cumulative prob ≤ 0.95
    "repeat_penalty": 1.1,   # penalizes repeated tokens to avoid repetition
    "num_predict": 1500,       # max number of tokens the model will generate
}
 
N_ENTRIES = 12
CLONES_PER_ENTRY = 1    
FUNCTION_NAME = "task_func"
REMOTE_OLLAMA = True
cg.FUNCTION_NAME = FUNCTION_NAME
cg.REMOTE_OLLAMA = REMOTE_OLLAMA

### Test connection to server

In [4]:
from src.clone_gen import call_ollama_chat
messages = [
    {"role": "user", "content": "Are you up and running, answer in one word."}
] 
response = call_ollama_chat(messages, OLLAMA_MODEL, LLM_OPTS)
print(response)

Yes!


## Generation of clones

### Generation of Additional Fields

In [5]:
# from src.clone_gen import add_generated_fields

# add_generated_fields(
#     dataset_path=DATASET_PATH,
#     nl_model=NL_MODEL,
#     llm_opts=LLM_OPTS,
#     n_entries=N_ENTRIES
# )

### Translating Source code 

In [6]:
# from src.clone_gen import add_generated_translation

# add_generated_translation(
#     dataset_path=DATASET_PATH,
#     code_model=CODE_MODEL,
#     llm_opts=LLM_OPTS,
#     n_entries=N_ENTRIES,
#     language="Java"
# )

## Generating clones

In [ ]:
import random
from itertools import combinations
from src.clone_gen import run_clone_generation

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

CONTEXTS = ["test", "complete", "ast", "code"]
REFACS = [f"refac_{i}" for i in range(1, 8)]  # refac_1..refac_7
STRATEGIES = ["zero-shot", "cot"]

# --- Generate all unique combinations of size 1-3 ---
all_combinations = []
for k in range(1, 4):
    all_combinations.extend(combinations(REFACS, k))

# Shuffle once deterministically
random.seed(RANDOM_SEED)
random.shuffle(all_combinations)

# --- Fixed subset of combinations to use for all pairs ---
NUM_COMBINATIONS_TO_USE = 5  
subset_combinations = all_combinations[:NUM_COMBINATIONS_TO_USE]

# --- Iterate over strategy + context ---
for strategy in STRATEGIES:
    for context in CONTEXTS:
        for run_idx, refac_tuple in enumerate(subset_combinations, 1):
            selected_refacs = list(refac_tuple)
            print(f"\n=== Generating for strategy={strategy}, context={context}, refactorings: {selected_refacs} ===")
            run_clone_generation(
                dataset_path=DATASET_PATH,
                out_path=OUT_PATH,
                n_entries=N_ENTRIES,
                clones_per_entry=CLONES_PER_ENTRY,
                ollama_model=OLLAMA_MODEL,
                llm_opts=LLM_OPTS,
                context=context,
                refacs=selected_refacs,
                strategy=strategy,
            )


[('refac_1',), ('refac_2',), ('refac_3',), ('refac_4',), ('refac_5',), ('refac_6',), ('refac_7',), ('refac_1', 'refac_2'), ('refac_1', 'refac_3'), ('refac_1', 'refac_4'), ('refac_1', 'refac_5'), ('refac_1', 'refac_6'), ('refac_1', 'refac_7'), ('refac_2', 'refac_3'), ('refac_2', 'refac_4'), ('refac_2', 'refac_5'), ('refac_2', 'refac_6'), ('refac_2', 'refac_7'), ('refac_3', 'refac_4'), ('refac_3', 'refac_5'), ('refac_3', 'refac_6'), ('refac_3', 'refac_7'), ('refac_4', 'refac_5'), ('refac_4', 'refac_6'), ('refac_4', 'refac_7'), ('refac_5', 'refac_6'), ('refac_5', 'refac_7'), ('refac_6', 'refac_7'), ('refac_1', 'refac_2', 'refac_3'), ('refac_1', 'refac_2', 'refac_4'), ('refac_1', 'refac_2', 'refac_5'), ('refac_1', 'refac_2', 'refac_6'), ('refac_1', 'refac_2', 'refac_7'), ('refac_1', 'refac_3', 'refac_4'), ('refac_1', 'refac_3', 'refac_5'), ('refac_1', 'refac_3', 'refac_6'), ('refac_1', 'refac_3', 'refac_7'), ('refac_1', 'refac_4', 'refac_5'), ('refac_1', 'refac_4', 'refac_6'), ('refac_1', 